# Parcel ISPC — Condition Contrasts (Left-Wing Subjects)

Compare ISPC values across conditions for parcels that are **FDR-significant and ISPC > 0**
in at least one condition (from `parcel_ispc_leftwing.ipynb` permutation test).

**Three contrasts:**
1. Agreed (AntiRight + ProLeft) vs Disagreed (AntiLeft + ProRight)
2. Within Agreed: AntiRight vs ProLeft
3. Within Disagreed: AntiLeft vs ProRight

**Two levels:**
- **Section A — Subject level**: paired t-test on LOO-ISPC per subject, FDR-corrected across parcels
- **Section B — Group mean**: descriptive comparison of group mean ISPC across conditions

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import fdrcorrection
import sys
from IPython.display import display

## Configuration

In [ ]:
ROOT         = Path('/path/to/project/root')  # CHANGE THIS to your local path
OUTPUT_DIR   = ROOT / 'data/derivatives/parcel_ispc/leftwing'
CONTRAST_DIR = OUTPUT_DIR / 'contrasts'
CONTRAST_DIR.mkdir(parents=True, exist_ok=True)

ATLAS_NII  = ROOT / 'data/atlases/Schaefer2018_tf_2mm_400Parcels7Networks_plus_TianS3.dseg.nii.gz'
LABELS_TSV = ROOT / 'data/atlases/Schaefer2018_400Parcels7Networks_plus_TianS3_labels.tsv'

sys.path.insert(0, str(ROOT / 'yy-fMRI-kit/src'))
from yy_fmri_kit.event_isc.contrast import parcels_to_nifti

RUN_TYPES      = ['AntiLeft', 'AntiRight', 'ProLeft', 'ProRight']
AGREED         = ['AntiRight', 'ProLeft']
DISAGREED      = ['AntiLeft',  'ProRight']
FDR_Q          = 0.05
ISPC_ABS_THRESH = 0.1   # parcel included if |isc_mean| >= this in >=1 condition of the contrast

## Load Data

Loads the significance table and per-subject ISPC matrices saved by `parcel_ispc_leftwing.ipynb`.

In [ ]:
# Significance table (one row per condition x parcel)
sig_df = pd.read_csv(OUTPUT_DIR / 'parcel_isc_B_significance.csv')
print(f'Significance table: {sig_df.shape}  conditions: {sig_df.condition.unique().tolist()}')

# Per-subject ISPC matrices (rows=subjects, cols=parcels)
isc_subj = {}
isc_mean = {}

for cond in RUN_TYPES:
    csv_path = OUTPUT_DIR / f'parcel_isc_B_{cond}_persubject.csv'
    df = pd.read_csv(csv_path, index_col=0)
    isc_subj[cond] = df
    print(f'  {cond}: {df.shape[0]} subjects x {df.shape[1]} parcels')

# Parcel name list
ALL_PARCEL_NAMES = list(isc_subj[RUN_TYPES[0]].columns)

# Align to common subjects
common_subjects = sorted(
    set(isc_subj[RUN_TYPES[0]].index)
    .intersection(*[set(isc_subj[c].index) for c in RUN_TYPES[1:]])
)
print(f'\nCommon subjects: {len(common_subjects)}')

# Re-index and convert to numpy
subj_arr = {}   # dict[cond] -> np.ndarray (n_subjects, n_parcels)
mean_arr = {}   # dict[cond] -> np.ndarray (n_parcels,)
for cond in RUN_TYPES:
    arr = isc_subj[cond].loc[common_subjects, ALL_PARCEL_NAMES].to_numpy(dtype=np.float64)
    subj_arr[cond] = arr
    mean_arr[cond] = np.nanmean(arr, axis=0)

## Parcel Selection

Parcel selection is **per contrast**, not global. For each contrast, include parcels that satisfy
both criteria in the conditions entering that contrast:

1. **FDR-significant** (`p_fdr < 0.05`) in at least one of the contrast's conditions
2. **|ISPC| ≥ 0.1** in at least one of those same conditions

A parcel only needs to pass either criterion in *any one* condition — it does not need to be
significant in both conditions simultaneously.

In [ ]:
# Overview: how many parcels are FDR-significant per condition
for cond in RUN_TYPES:
    n_sig = (sig_df[(sig_df['condition'] == cond) & (sig_df['significant'] == True)]).shape[0]
    n_thr = (sig_df[(sig_df['condition'] == cond) & (sig_df['isc_mean'].abs() >= ISPC_ABS_THRESH)]).shape[0]
    print(f'{cond}: {n_sig} FDR-sig parcels | {n_thr} parcels with |ISPC| >= {ISPC_ABS_THRESH}')

print('\nPer-contrast parcel counts are shown when each contrast is run below.')

## Helper Functions

In [ ]:
def get_parcels_for_contrast(conditions):
    """
    Select parcels that are:
      (1) FDR-significant in >=1 of the given conditions, AND
      (2) |isc_mean| >= ISPC_ABS_THRESH in >=1 of those same conditions.
    """
    cond_mask = sig_df['condition'].isin(conditions)
    sig_set   = set(sig_df.loc[cond_mask & (sig_df['significant'] == True), 'parcel_name'])
    thr_set   = set(sig_df.loc[cond_mask & (sig_df['isc_mean'].abs() >= ISPC_ABS_THRESH), 'parcel_name'])
    return sorted(sig_set & thr_set)


def run_contrast(a_arr, b_arr, label_a, label_b, parcel_list):
    """Paired t-test for each parcel in parcel_list; returns DataFrame sorted by p."""
    rows = []
    for pname in parcel_list:
        idx  = ALL_PARCEL_NAMES.index(pname)
        a, b = a_arr[:, idx], b_arr[:, idx]
        mask = ~(np.isnan(a) | np.isnan(b))
        a_c, b_c = a[mask], b[mask]
        if len(a_c) < 3:
            continue
        t, p      = stats.ttest_rel(a_c, b_c)
        diff      = a_c - b_c
        mean_diff = float(np.mean(diff))
        cohen_d   = mean_diff / (float(np.std(diff, ddof=1)) + 1e-12)
        rows.append({
            'parcel_name'     : pname,
            f'mean_{label_a}' : round(float(np.mean(a_c)), 4),
            f'mean_{label_b}' : round(float(np.mean(b_c)), 4),
            'mean_diff (A-B)' : round(mean_diff, 4),
            'cohen_d'         : round(cohen_d, 3),
            't'               : round(float(t), 3),
            'p_raw'           : round(float(p), 5),
            'n_subjects'      : int(mask.sum()),
        })
    df = pd.DataFrame(rows)
    if len(df) > 0:
        _, p_fdr = fdrcorrection(df['p_raw'].values, alpha=FDR_Q)
        df['p_fdr']       = np.round(p_fdr, 5)
        df['significant'] = df['p_fdr'] < FDR_Q
        df = df.sort_values('p_raw').reset_index(drop=True)
    return df


def box_plot(result_df, a_arr, b_arr, label_a, label_b,
             color_a='#5B8DB8', color_b='#6AA96B', title='', out_path=None):
    """Box plot with paired subject dots for FDR-sig parcels (top 5 fallback)."""
    plot_df = result_df[result_df['significant']]
    if len(plot_df) == 0:
        plot_df = result_df.head(5)
        print('  No FDR-significant parcels — showing top 5 by p-value')

    n   = len(plot_df)
    fig, axes = plt.subplots(1, n, figsize=(max(5, n * 3.0), 4.5), squeeze=False)
    rng = np.random.default_rng(42)

    for i, (_, row) in enumerate(plot_df.iterrows()):
        ax     = axes[0, i]
        idx    = ALL_PARCEL_NAMES.index(row['parcel_name'])
        a_vals = a_arr[:, idx]
        b_vals = b_arr[:, idx]

        # Box plots
        bp = ax.boxplot(
            [a_vals, b_vals],
            positions=[0, 1], widths=0.45, patch_artist=True,
            medianprops=dict(color='black', linewidth=2),
            boxprops=dict(linewidth=1.2),
            whiskerprops=dict(linewidth=1.0),
            capprops=dict(linewidth=1.0),
            flierprops=dict(marker=''),
        )
        for patch, col in zip(bp['boxes'], [color_a, color_b]):
            patch.set_facecolor(col)
            patch.set_alpha(0.65)

        # Individual subject dots
        for xi, vals in enumerate([a_vals, b_vals]):
            jitter = rng.uniform(-0.08, 0.08, size=len(vals))
            ax.scatter(xi + jitter, vals, color='black', s=16, alpha=0.55, zorder=3)

        # Paired lines
        for av, bv in zip(a_vals, b_vals):
            ax.plot([0, 1], [av, bv], color='grey', alpha=0.20, linewidth=0.8)

        # Significance bracket
        p_fdr = row['p_fdr']
        star  = '***' if p_fdr < 0.001 else '**' if p_fdr < 0.01 else '*' if p_fdr < 0.05 else 'n.s.'
        y_top = max(np.nanmax(a_vals), np.nanmax(b_vals)) + 0.03
        ax.plot([0, 0, 1, 1], [y_top, y_top+0.01, y_top+0.01, y_top], color='black', linewidth=1)
        ax.text(0.5, y_top + 0.012, star, ha='center', va='bottom', fontsize=10)

        # Title
        parts = row['parcel_name'].split('_')
        short = ('_'.join(parts[-4:-2]) + '\n' + '_'.join(parts[-2:])) \
                if len(parts) >= 4 else row['parcel_name']
        ax.set_title(short, fontsize=8, pad=3, linespacing=1.3)
        ax.set_xticks([0, 1])
        ax.set_xticklabels([label_a, label_b], fontsize=8)
        ax.set_ylabel('LOO-ISPC', fontsize=8)
        ax.tick_params(labelsize=7)
        ax.text(0.97, 0.03,
                f'p_fdr={p_fdr:.3f}\nd={row["cohen_d"]:.2f}',
                transform=ax.transAxes, ha='right', va='bottom', fontsize=7)

    fig.suptitle(title, fontsize=11, y=1.02)
    plt.tight_layout()
    if out_path:
        plt.savefig(str(out_path), dpi=150, bbox_inches='tight')
    plt.show()
    return fig

---
## Section A — Subject-Level Paired t-Tests

One LOO-ISPC value per subject per condition; paired t-test across subjects.
FDR correction (BH, q = 0.05) applied across all selected parcels within each contrast.

### Contrast 1 — Agreed vs Disagreed

**Agreed** = (AntiRight + ProLeft) / 2 per subject  
**Disagreed** = (AntiLeft + ProRight) / 2 per subject  
Positive `mean_diff` = higher ISPC for agreed content.

In [ ]:
# Parcels: FDR-sig AND |ISPC| >= 0.1 in any of the 4 conditions
parcels_c1 = get_parcels_for_contrast(['AntiLeft', 'AntiRight', 'ProLeft', 'ProRight'])
print(f'Parcels selected for Contrast 1: {len(parcels_c1)}')

agreed_subj    = (subj_arr['AntiRight'] + subj_arr['ProLeft'])  / 2
disagreed_subj = (subj_arr['AntiLeft']  + subj_arr['ProRight']) / 2

c1 = run_contrast(agreed_subj, disagreed_subj, 'Agreed', 'Disagreed', parcels_c1)
print(f'Contrast 1 — Agreed vs Disagreed  |  {len(c1)} parcels tested  |  {c1["significant"].sum()} FDR-sig')
display(c1.head(15))

In [ ]:
box_plot(
    c1, agreed_subj, disagreed_subj, 'Agreed', 'Disagreed',
    color_a='#5B8DB8', color_b='#E07B54',
    title='Contrast 1: Agreed vs Disagreed — Subject-Level LOO-ISPC',
    out_path=CONTRAST_DIR / 'subj_contrast1_agreed_vs_disagreed.png',
)

### Contrast 2 — AntiRight vs ProLeft (within Agreed)

Tests whether the two agreed sub-types differ in neural synchrony.

In [ ]:
# Parcels: FDR-sig AND |ISPC| >= 0.1 in AntiRight or ProLeft
parcels_c2 = get_parcels_for_contrast(['AntiRight', 'ProLeft'])
print(f'Parcels selected for Contrast 2: {len(parcels_c2)}')

c2 = run_contrast(subj_arr['AntiRight'], subj_arr['ProLeft'], 'AntiRight', 'ProLeft', parcels_c2)
print(f'Contrast 2 — AntiRight vs ProLeft  |  {len(c2)} parcels tested  |  {c2["significant"].sum()} FDR-sig')
display(c2.head(15))

In [ ]:
box_plot(
    c2, subj_arr['AntiRight'], subj_arr['ProLeft'], 'AntiRight', 'ProLeft',
    color_a='#5B8DB8', color_b='#6AA96B',
    title='Contrast 2: AntiRight vs ProLeft — Subject-Level LOO-ISPC',
    out_path=CONTRAST_DIR / 'subj_contrast2_antiright_vs_proleft.png',
)

### Contrast 3 — AntiLeft vs ProRight (within Disagreed)

Tests whether the two disagreed sub-types differ in neural synchrony.

In [ ]:
# Parcels: FDR-sig AND |ISPC| >= 0.1 in AntiLeft or ProRight
parcels_c3 = get_parcels_for_contrast(['AntiLeft', 'ProRight'])
print(f'Parcels selected for Contrast 3: {len(parcels_c3)}')

c3 = run_contrast(subj_arr['AntiLeft'], subj_arr['ProRight'], 'AntiLeft', 'ProRight', parcels_c3)
print(f'Contrast 3 — AntiLeft vs ProRight  |  {len(c3)} parcels tested  |  {c3["significant"].sum()} FDR-sig')
display(c3.head(15))

In [ ]:
box_plot(
    c3, subj_arr['AntiLeft'], subj_arr['ProRight'], 'AntiLeft', 'ProRight',
    color_a='#E07B54', color_b='#9B6BB5',
    title='Contrast 3: AntiLeft vs ProRight — Subject-Level LOO-ISPC',
    out_path=CONTRAST_DIR / 'subj_contrast3_antileft_vs_proright.png',
)

---
## Brain Maps — FDR-Significant Parcels per Contrast

Yabplot cortical surface maps and subcortical volume slices for each contrast.
Parcels are coloured by their **t-statistic** (direction + magnitude of effect);
non-significant parcels are masked grey.

- **Red** → first condition > second
- **Blue** → second condition > first
- Subcortical Tian-S3 parcels shown as axial nilearn slices below the surface map.

In [ ]:
import yabplot as yab
import yabplot.data as ydata
import nibabel as nib
import tempfile, os
from nilearn import plotting

BRAIN_MAP_DIR = CONTRAST_DIR / 'brain_maps_contrast'
BRAIN_MAP_DIR.mkdir(exist_ok=True)

ALL_VIEWS = [
    'left_lateral', 'left_medial', 'right_lateral', 'right_medial',
    'superior', 'inferior', 'anterior', 'posterior',
]

try:
    _lh_surf, _rh_surf = ydata.get_surface_paths('midthickness', 'bmesh')
except Exception as e:
    print(f'yabplot surface load warning: {e}')
    _lh_surf = _rh_surf = None


def _sig_arr(result_df, value_col='t'):
    """Full-atlas array with value for FDR-sig parcels, NaN elsewhere."""
    arr = np.full(len(ALL_PARCEL_NAMES), np.nan)
    for _, row in result_df[result_df['significant']].iterrows():
        pname = row['parcel_name']
        if pname in ALL_PARCEL_NAMES:
            arr[ALL_PARCEL_NAMES.index(pname)] = row[value_col]
    return arr


def _yab_cortical(nii_path, out_png, vminmax):
    lh, rh = yab.project_vol2surf(str(nii_path), interpolation='nearest')
    lm, rm = yab.load_vertexwise_mesh(_lh_surf, _rh_surf, lh, rh)
    yab.plot_vertexwise(
        lm, rm, views=ALL_VIEWS, cmap='RdBu_r',
        vminmax=vminmax, figsize=(1600, 800),
        display_type='static', export_path=str(out_png),
        nan_color=(0.92, 0.92, 0.92),
    )
    return out_png


def _nilearn_subcortical(nii_path, out_png, vmax, title=''):
    disp = plotting.plot_stat_map(
        str(nii_path),
        display_mode='z', cut_coords=8,
        cmap='RdBu_r', vmax=vmax, symmetric_cbar=True,
        title=title, draw_cross=False,
    )
    disp.savefig(str(out_png), dpi=150)
    disp.close()
    return out_png

In [ ]:
CONTRASTS = [
    (c1, 'agreed_vs_disagreed',  'Agreed vs Disagreed'),
    (c2, 'antiright_vs_proleft', 'AntiRight vs ProLeft'),
    (c3, 'antileft_vs_proright', 'AntiLeft vs ProRight'),
]

for df, cname, label in CONTRASTS:
    n_sig = int(df['significant'].sum())
    print(f'\n{"="*55}')
    print(f'{label}  |  {n_sig} FDR-significant parcels')

    if n_sig == 0:
        print('  (nothing to plot)')
        continue

    arr  = _sig_arr(df, value_col='t')
    vext = max(float(np.nanmax(np.abs(arr))), 0.01)

    tmp = tempfile.mktemp(suffix='.nii.gz')
    parcels_to_nifti(arr, ALL_PARCEL_NAMES, ATLAS_NII, LABELS_TSV, tmp)

    # Cortical surface — yabplot
    if _lh_surf is not None:
        out_png = BRAIN_MAP_DIR / f'{cname}_cortical.png'
        _yab_cortical(tmp, out_png, vminmax=[-vext, vext])
        print(f'  Cortical   -> {out_png.name}')
    else:
        print('  (yabplot surfaces not available, skipping cortical map)')

    # Subcortical — identify Tian-S3 parcels (no '7Networks_' prefix)
    sig_names = df[df['significant']]['parcel_name'].tolist()
    subcort_sig = [p for p in sig_names if not p.startswith('7Networks_')]
    print(f'  Subcortical FDR-sig parcels ({len(subcort_sig)}): {subcort_sig}')

    out_sub = BRAIN_MAP_DIR / f'{cname}_subcortical.png'
    _nilearn_subcortical(tmp, out_sub, vmax=vext,
                         title=f'{label} — t-stat (subcortical)')
    print(f'  Subcortical -> {out_sub.name}')

    os.unlink(tmp)

## Save Results

In [ ]:
for cname, df in [
    ('agreed_vs_disagreed',  c1),
    ('antiright_vs_proleft', c2),
    ('antileft_vs_proright', c3),
]:
    out = CONTRAST_DIR / f'contrast_{cname}_subject_level.csv'
    df.to_csv(out, index=False)
    print(f'{cname}: {len(df)} parcels, {df["significant"].sum()} FDR-sig -> {out.name}')

mean_df.to_csv(CONTRAST_DIR / 'group_mean_per_parcel.csv', index=False)
print('\ngroup_mean_per_parcel.csv saved')